In [4]:
%pip install pymongo pandas

Note: you may need to restart the kernel to use updated packages.


In [5]:
import sys 
from pathlib import Path

project_root = None
for p in Path.cwd().resolve().parents:
    if (p / "utils").exists() and (p / "data").exists():
        project_root = p
        break

if project_root is None:
    raise RuntimeError("Raíz no encontrada.")

sys.path.insert(0, str(project_root))

In [ ]:
import os
import certifi
import pymongo
import logging
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from pymongo import MongoClient
from utils.paths import find_project_root

logging.basicConfig(
	level=logging.INFO,
	format="%(asctime)s | %(levelname)s | %(message)s"
)


root = find_project_root()

csv_path = root / "data" / "raw" / "job_market.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)

logging.info(f"Ruta {root} encontrada.")
logging.info(f"CSV exportado hacia {csv_path}")

load_dotenv()

MONGODB_URI = os.getenv("MONGODB_URI")
if not MONGODB_URI:
	logging.error("No se encontró MONGODB_URI en las variables de entorno.")
	raise RuntimeError("MONGODB_URI is not defined in the environment variables.")

logging.info("Variable MONGODB_URI cargada.")

try:
	client = MongoClient(MONGODB_URI, tlsCAFile=certifi.where())
	db = client["job_market"]
	collection = db["jobs"]
	logging.info("Conexión a MongoDB completada")
except Exception as e:
    logging.exception("Error en la conexión")
    raise e

try:
	data = list(collection.find({}))
	logging.info(f"Se recuperaron {len(data)} documentos.")
except Exception as e:
    logging.exception("Error en la consulta a la base.")



df = pd.DataFrame(data)

if "_id" in df.columns:
	df.drop(columns=["_id"], inplace=True)
	logging.info("Columna _id eliminada.")

df.to_csv(csv_path, index=False, encoding="utf-8")
logging.info(f"CSV creado en {csv_path.relative_to(root)}")

2025-12-02 11:17:10,864 | INFO | Ruta C:\Users\gianlu\Market-Scraper\Market-Scraper encontrada.
2025-12-02 11:17:10,865 | INFO | CSV exportado hacia C:\Users\gianlu\Market-Scraper\Market-Scraper\data\raw\job_market.csv
2025-12-02 11:17:10,866 | INFO | Variable MONGODB_URI cargada.
2025-12-02 11:17:11,319 | INFO | Conexión a MongoDB completada
